# 02 — Silver Transformation

Reads from Bronze Delta table, validates data quality, reshapes to long format,
and writes to Silver. Depends on `01_ingest` completing successfully.

In [ ]:
%pip install pandas numpy -q

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
CATALOG = 'workspace'
SCHEMA  = 'default'

## Section 3: Build Segmented Demand Table

Reshape wide monthly Trends table into long format (one row per date x segment),
then resample from monthly to daily via forward fill.

In [ ]:
# Read from Bronze Delta table — Silver does not depend on pandas memory
df_trends_bronze = spark.read.table(f'{CATALOG}.{SCHEMA}.bronze_trends').toPandas()
df_trends_bronze['date'] = pd.to_datetime(df_trends_bronze['date'])

# Restore segment names (underscores were added for Delta compatibility)
df_trends_bronze.columns = [c.replace('_', ' ') if c != 'date' else c
                             for c in df_trends_bronze.columns]

# Restore numeric types (Bronze stored everything as string)
seg_cols = [c for c in df_trends_bronze.columns if c != 'date']
for col in seg_cols:
    df_trends_bronze[col] = pd.to_numeric(df_trends_bronze[col], errors='coerce')

# Monthly -> daily via forward fill
df_daily_wide = (
    df_trends_bronze
    .set_index('date')
    .resample('D')
    .ffill()
    .reset_index()
)

# Wide -> long
df_long = df_daily_wide.melt(
    id_vars='date',
    value_vars=seg_cols,
    var_name='segment',
    value_name='demand'
).dropna(subset=['demand'])

df_long['date']  = pd.to_datetime(df_long['date'])
df_long['level'] = df_long['segment'].str.split().str[0]

print(f'Read from {CATALOG}.{SCHEMA}.bronze_trends')
print(f'Long-format table: {len(df_long):,} rows, {df_long.segment.nunique()} segments')
print(f'Date range: {df_long.date.min().date()} to {df_long.date.max().date()}')
display(spark.createDataFrame(df_long.head(10).astype(str)))

### Silver Layer — Validated & Cleaned

Read from Bronze, apply data quality checks, reshape to long format.
Rows that fail checks are flagged — the pipeline raises an error rather than silently producing bad output.

In [ ]:
# --- Data quality checks ---
assert df_long['date'].isna().sum() == 0,     'Silver check failed: null dates detected'
assert df_long['demand'].between(0, 100).all(),     'Silver check failed: demand values outside valid 0-100 range'
assert df_long['segment'].nunique() == 11,     f'Silver check failed: expected 11 segments, got {df_long["segment"].nunique()}'

# Warn if any segment has a date gap > 60 days (missing months)
for seg, grp in df_long.groupby('segment'):
    max_gap = grp['date'].sort_values().diff().dt.days.dropna().max()
    if max_gap > 60:
        print(f'WARNING: {seg} has a date gap of {int(max_gap)} days')

print('Silver quality checks passed.')

# Write to Silver Delta table
(spark.createDataFrame(df_long)
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.silver_demand'))

print(f'Silver table written: {CATALOG}.{SCHEMA}.silver_demand ({len(df_long):,} rows)')